In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tugas4_PySpark").master("local[*]").getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession Tugas 4 berhasil dibuat!")

26/09/10 18:38:12 WARN Utils: Your hostname, raam resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/10 18:38:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 18:38:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/10 18:38:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession Tugas 4 berhasil dibuat!


In [10]:
#Membaca dan Eksplorasi Awal
df_tugas = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)
print("Skema Data; ")
df_tugas.printSchema()

print("Jumlah total baris:", df_tugas.count())

print("\n 10 Baris Pertama: ")
df_tugas.show(10)

Skema Data; 
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris: 1000

 10 Baris Pertama: 
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|

In [11]:
#Menangani Data Kosong
from pyspark.sql.functions import col

jumlah_null = df_tugas.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_null}")

df_clean = df_tugas.na.fill({"rating": 0})

print("Jumlah rating null setelah penanganan:", df_clean.filter(col("rating").isNull()).count())

Jumlah baris dengan rating kosong: 204
Jumlah rating null setelah penanganan: 0


**Pilih salah satu antara df.na.fill() atau df.na.drop(), jelaskan alasannya pada markdown cell) untuk menanganinya.**
Alasan saya memilih mengisi rating yang kosong dengan angka 0 menggunakan na.fill() daripada menghapus barisnya dengan (na.drop()). Karena jika barisnya dihapus, data transaksi lain yang masih lengkap (seperti jumlah unit dan harga) juga ikut terbuang, jadi hitungan total pendapatan nanti bisa tidak akurat.

In [5]:
#Transformasi Data
from pyspark.sql.functions import col, when

df_transformed = df_clean.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga_satuan")
).withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [7]:
#Analisis dengan GroupBy
from pyspark.sql.functions import sum as spark_sum, count, avg, round as spark_round, col

# 1. Kategori dengan total_pendapatan tertinggi
print("--- 1. Kategori dengan Total Pendapatan Tertinggi ---")
df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("--- 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ---")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("--- 3. Rata-rata Rating per Metode Pembayaran ---")
df_transformed.groupBy("metode_pembayaran") \
    .agg(spark_round(avg("rating"), 2).alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

--- 1. Kategori dengan Total Pendapatan Tertinggi ---
+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

--- 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ---
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

--- 3. Rata-rata Rating per Metode Pembayaran ---
+-----------------+----------------+
|metode_pembayaran|rata_rata_rating|
+-----------------+----------------+
|              COD|            3.37|
|    Transfer Bank|            3.34|
|         E-Wallet|            3.29|
|     Kartu Kredit|            3.19|
+-----------------+----------------+



In [8]:
#Menyimpan Hasil ke HDFS 
path_output = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_pemrosesan_september"

df_transformed.write.mode("overwrite").option("header", "true").csv(path_output)

print("Data berhasil disimpan ke HDFS!")

!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_pemrosesan_september

Data berhasil disimpan ke HDFS!
Found 2 items
-rw-r--r--   3 raamxh supergroup          0 2026-09-10 18:40 /user/mahasiswa/tugas4/hasil_pemrosesan_september/_SUCCESS
-rw-r--r--   3 raamxh supergroup      97296 2026-09-10 18:40 /user/mahasiswa/tugas4/hasil_pemrosesan_september/part-00000-99956bb2-52a2-48c3-a5fc-333e25cd3d14-c000.csv


**Spark menyimpan hasil sebagai beberapa berkas partisi (part-00000..., dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi**
Karena Spark bekerja secara terdistribusi/paralel. Jadi saat menyimpan data ke HDFS, setiap worker di Spark akan menulis bagian datanya masing-masing secara bersamaan. Cara ini jauh lebih cepat dibanding harus menggabungkan semua data menjadi satu file dulu baru disimpan.


In [12]:
spark.stop()
print("SparkSession berhasil ditutup.")

SparkSession berhasil ditutup.
